[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/badaouihakimou/machine-learning-notebooks/blob/main/08_modules_de_base.ipynb)



# Les modules

Les fonctions intégrées du notebook précédent sont disponibles sans rien faire.
Tout le reste vit dans des modules, qu'il faut importer.

Un module est un simple fichier `.py` contenant du code. Un package est un
dossier qui en regroupe plusieurs.

## Le plan

| Section | Le sujet |
|---|---|
| 1 | Écrire son propre module |
| 2 | Les quatre formes d'import |
| 3 | Ce qui se passe vraiment à l'import |
| 4 | `math` |
| 5 | `statistics`, et le piège de la variance |
| 6 | `random`, et la reproductibilité |
| 7 | `os` et `glob` |

## Deux pièges annoncés

`statistics.variance` ne calcule pas la même chose que `np.var`. Les deux sont
justes, mais ils ne répondent pas à la même question.

Et `random` sans graine rend un notebook non reproductible : le lecteur obtiendra
des résultats différents de ceux affichés.

Prérequis : les notebooks 02 à 07.

## 1. Écrire son propre module

La magie `%%writefile` écrit le contenu d'une cellule dans un fichier. C'est le
moyen de créer un module depuis un notebook.

Elle doit être en toute première ligne de la cellule, sans commentaire
au-dessus. Certains environnements tolèrent une ligne avant, d'autres non — ne
prends pas le risque.

In [1]:
%%writefile mes_outils.py

def fibonacci(n):
    """Retourne les termes de la suite de Fibonacci strictement inférieurs à n."""
    a, b = 0, 1
    liste = []
    while a < n:
        liste.append(a)
        a, b = b, a + b
    return liste


def trier(classeur, nombre):
    """Range un nombre dans le classeur, selon son signe.

    Le classeur doit contenir les clés 'positive' et 'negative'.
    Modifie le classeur passé en argument.
    """
    if nombre > 0:
        classeur['positive'].append(nombre)
    else:
        classeur['negative'].append(nombre)
    return classeur


VERSION = '1.0'

Writing mes_outils.py


Vérifions que le fichier existe et contient bien ce qu'on attend.

In [2]:
import os
print('Fichier présent :', os.path.exists('mes_outils.py'))
print()
with open('mes_outils.py') as f:
    print(f.read()[:200], '...')

Fichier présent : True


def fibonacci(n):
    """Retourne les termes de la suite de Fibonacci strictement inférieurs à n."""
    a, b = 0, 1
    liste = []
    while a < n:
        liste.append(a)
        a, b = b, a + b
   ...


### L'utiliser

In [3]:
import mes_outils

print(mes_outils.fibonacci(10))
print(mes_outils.VERSION)

classeur = {'positive': [], 'negative': []}
for n in [3, -5, 0, 12, -1]:
    mes_outils.trier(classeur, n)

print(classeur)

[0, 1, 1, 2, 3, 5, 8]
1.0
{'positive': [3, 12], 'negative': [-5, 0, -1]}


Deux remarques sur `fibonacci`.

Le `a, b = b, a + b` est l'échange simultané du notebook 02 : le côté droit est
entièrement évalué avant toute affectation. Sans lui, il faudrait une variable
temporaire.

Et le paramètre `n` est une borne, pas un nombre de termes. `fibonacci(10)`
donne les termes inférieurs à 10, soit sept valeurs. La docstring lève
l'ambiguïté c'est exactement à ça qu'elle sert.

Sur `trier`, un point de conception vu au notebook 04 : la fonction modifie
le dictionnaire reçu. Le `return` est donc un peu trompeur, puisqu'il renvoie le
même objet.

In [4]:
resultat = mes_outils.trier(classeur, 7)
print('Même objet ?', resultat is classeur)

Même objet ? True


## 2. Les quatre formes d'import

In [5]:
import mes_outils # 1. tout, préfixé
import mes_outils as mo # 2. avec un alias
from mes_outils import fibonacci # 3. une seule chose
from mes_outils import fibonacci as fibo # 4. avec un alias

print(mes_outils.fibonacci(10))
print(mo.fibonacci(10))
print(fibonacci(10))
print(fibo(10))

[0, 1, 1, 2, 3, 5, 8]
[0, 1, 1, 2, 3, 5, 8]
[0, 1, 1, 2, 3, 5, 8]
[0, 1, 1, 2, 3, 5, 8]


| Forme | Quand l'utiliser |
|---|---|
| `import module` | par défaut, l'origine reste visible |
| `import module as m` | quand le nom est long `np`, `pd`, `plt` |
| `from module import x` | quand on n'utilise qu'une ou deux choses |
| `from module import *` | jamais, ou presque |

Les alias `np`, `pd`, `plt` sont des conventions si répandues qu'il vaut mieux
les respecter tout le code que tu liras les utilise.

### Pourquoi éviter l'étoile

`from module import *` importe tout sans préfixe, et écrase silencieusement ce
qui porte le même nom.

In [6]:
VERSION = 'ma version à moi'
print('avant :', VERSION)

from mes_outils import *

print('après :', VERSION)

avant : ma version à moi
après : 1.0


Ta variable a disparu, sans le moindre message.

C'est le même mécanisme que `list = [1, 2, 3]` du notebook 06, mais en pire : ici
tu ne vois même pas quels noms arrivent. Avec deux `import *` successifs, le
second écrase le premier, et l'origine d'une fonction devient impossible à
retrouver.

L'autre problème est la lisibilité. En lisant `sqrt(x)` sans préfixe, on ne sait
pas si c'est `math.sqrt`, `numpy.sqrt` ou une fonction maison et les trois ne
se comportent pas pareil.

## 3. Ce qui se passe vraiment à l'import

Importer, c'est exécuter le fichier. Toute ligne au niveau zéro s'exécute.

In [7]:
%%writefile module_bavard.py

print('>>> je suis exécuté à l\'import')

def salut():
    return 'bonjour'

Writing module_bavard.py


In [8]:
import module_bavard

>>> je suis exécuté à l'import


In [9]:
import module_bavard
print('Deuxième import : aucun message')

Deuxième import : aucun message


Le message n'apparaît qu'une fois. Python garde les modules chargés dans
`sys.modules` et ne les réexécute pas.

In [10]:
import sys
print('module_bavard en cache :', 'module_bavard' in sys.modules)
print('Fichier source :', module_bavard.__file__)

module_bavard en cache : True
Fichier source : /content/module_bavard.py


### La conséquence : les modifications ne sont pas vues

C'est le piège pratique le plus pénible quand on développe un module en parallèle
d'un notebook.

In [11]:
%%writefile module_bavard.py

def salut():
    return 'BONJOUR EN MAJUSCULES'

Overwriting module_bavard.py


In [12]:
import module_bavard
print(module_bavard.salut()) # affiche encore l'ancienne version

bonjour


In [13]:
import importlib
importlib.reload(module_bavard)

print(module_bavard.salut()) # maintenant à jour

BONJOUR EN MAJUSCULES


`importlib.reload` force la relecture du fichier. Sans lui, on modifie son code
et on ne comprend pas pourquoi rien ne change.

Attention : `reload` ne met pas à jour les noms importés avec `from module import
x` ceux-là pointent toujours vers l'ancienne fonction. Il faut refaire l'import.

### Le dossier `__pycache__`

Après un import, un dossier `__pycache__` apparaît.

In [14]:
import glob
print(glob.glob('*'))

['module_bavard.py', 'mes_outils.py', '__pycache__', 'sample_data']


Python y range une version compilée du module, pour accélérer les imports
suivants. C'est automatique et sans conséquence.

En revanche, ce dossier n'a rien à faire sur GitHub c'est pour ça qu'il
figure dans le `.gitignore` du dépôt.

### Où Python cherche les modules

In [15]:
import sys
for chemin in sys.path[:4]:
    print(repr(chemin))

'/content'
'/env/python'
'/usr/lib/python312.zip'
'/usr/lib/python3.12'


Le premier élément, souvent une chaîne vide ou le dossier courant, explique que
`mes_outils.py` soit trouvé sans configuration.

Un piège classique en découle : nommer son fichier `random.py` ou `math.py` fait
que Python importe ton fichier au lieu du module standard. L'erreur qui suit
est parfaitement incompréhensible.

### La convention `__main__`

Pour qu'un fichier puisse être à la fois module et programme :

```python
def ma_fonction():
    ...

if __name__ == '__main__':
    print('exécuté directement')      # pas à l'import
```

`__name__` vaut `'__main__'` quand le fichier est lancé directement, et le nom du
module quand il est importé. C'est ce qui permet de mettre des tests dans un
module sans qu'ils s'exécutent chez celui qui l'importe.

## 4. Le module math

La liste complète des modules standards est sur
[docs.python.org/3/py-modindex.html](https://docs.python.org/3/py-modindex.html).

In [16]:
import math

print('pi :', math.pi)
print('e :', math.e)
print('inf :', math.inf)

pi : 3.141592653589793
e : 2.718281828459045
inf : inf


In [17]:
print('cos(pi) :', math.cos(math.pi))
print('cos(2pi) :', math.cos(2 * math.pi))
print('exp(5) :', math.exp(5))
print('sqrt(16) :', math.sqrt(16))
print('log(100, 10) :', math.log(100, 10))
print('factorial(5) :', math.factorial(5))

cos(pi) : -1.0
cos(2pi) : 1.0
exp(5) : 148.4131591025766
sqrt(16) : 4.0
log(100, 10) : 2.0
factorial(5) : 120


### Arrondir dans une direction précise

`round` arrondit au plus proche, avec l'arrondi au pair du notebook 07. `math`
propose les deux directions strictes.

In [18]:
for x in [2.1, 2.5, 2.9, -2.1, -2.9]:
    print(f'{x:>5} : floor={math.floor(x):>3}  ceil={math.ceil(x):>3}  '
          f'trunc={math.trunc(x):>3}  round={round(x):>3}')

  2.1 : floor=  2  ceil=  3  trunc=  2  round=  2
  2.5 : floor=  2  ceil=  3  trunc=  2  round=  2
  2.9 : floor=  2  ceil=  3  trunc=  2  round=  3
 -2.1 : floor= -3  ceil= -2  trunc= -2  round= -2
 -2.9 : floor= -3  ceil= -2  trunc= -2  round= -3


`floor` arrondit vers le bas, `ceil` vers le haut, `trunc` vers zéro. Sur les
nombres négatifs, `floor` et `trunc` diffèrent : `floor(-2.1)` donne -3, `trunc`
donne -2.

### Le piège des flottants, à nouveau

`math.isclose` est la bonne façon de comparer deux flottants, comme vu au
notebook 02.

In [19]:
print(0.1 + 0.2 == 0.3)
print(math.isclose(0.1 + 0.2, 0.3))
print('cos(pi/2) devrait valoir 0 :', math.cos(math.pi / 2))
print('isclose de 0 :', math.isclose(math.cos(math.pi / 2), 0, abs_tol=1e-9))

False
True
cos(pi/2) devrait valoir 0 : 6.123233995736766e-17
isclose de 0 : True


`cos(pi/2)` ne donne pas exactement zéro mais 6e-17, parce que `math.pi` est une
approximation. Sur une comparaison à zéro, il faut `abs_tol` le `rel_tol` par
défaut ne fonctionne pas près de zéro.

Note pour la suite : `math` ne travaille que sur des nombres isolés.

```python
math.sqrt([1, 4, 9]) # TypeError
np.sqrt([1, 4, 9]) # array([1., 2., 3.])
```

Dès qu'il s'agit de tableaux, c'est NumPy qu'il faut.

In [45]:
math.sqrt([1, 4, 9]) # TypeError
np.sqrt([1, 4, 9]) # array([1., 2., 3.])

## 5. Le module statistics

In [21]:
import statistics
notes = [2, 3, 4, 5, 5, 6, 7, 76]
print('moyenne :', statistics.mean(notes))
print('médiane :', statistics.median(notes))
print('mode :', statistics.mode(notes))
print('écart-type:', round(statistics.stdev(notes), 3))

moyenne : 13.5
médiane : 5.0
mode : 5
écart-type: 25.304


La moyenne vaut 13,5 alors que la médiane vaut 5. L'écart vient du 76, une valeur
extrême qui tire la moyenne vers le haut sans affecter la médiane.

C'est la raison pour laquelle on utilise la médiane pour imputer des valeurs
manquantes, comme dans le notebook Pandas.

### Le piège de la variance

Il existe deux variances, et `statistics` propose les deux.

In [22]:
print('variance :', statistics.variance(notes))
print('pvariance :', statistics.pvariance(notes))

variance : 640.2857142857143
pvariance : 560.25


Deux valeurs différentes pour les mêmes données. Ce n'est pas une erreur.

`variance` divise par n-1 : c'est la variance d'échantillon, utilisée
quand les données sont un tirage parmi une population plus large dont on veut
estimer la variance.

`pvariance` divise par n : c'est la variance de population, quand les
données constituent l'ensemble complet.

Le `n-1` s'appelle la correction de Bessel. Elle compense le fait qu'un
échantillon sous-estime naturellement la dispersion de la population dont il est
issu.

Et voici le piège : NumPy fait le choix inverse par défaut.

In [23]:
import numpy as np
print('statistics.variance :', statistics.variance(notes))
print('numpy var (défaut) :', np.var(notes))
print('numpy var (ddof=1) :', np.var(notes, ddof=1))

statistics.variance : 640.2857142857143
numpy var (défaut) : 560.25
numpy var (ddof=1) : 640.2857142857143


`np.var` divise par n, `statistics.variance` par n-1. Deux bibliothèques
standards, deux conventions opposées.

Sur huit valeurs, l'écart est de 14 %. Sur mille, il devient négligeable. Mais si
tu compares un résultat obtenu avec `statistics` à un autre obtenu avec NumPy, tu
chercheras longtemps d'où vient la différence.

La règle : précise toujours `ddof` quand ça compte, et sache que Pandas suit
la convention de `statistics` `df.var()` utilise n-1.

`statistics` reste un module d'appoint. Sur de vraies données, NumPy et Pandas
sont bien plus rapides et complets.

## 6. Le module random

In [24]:
import random

liste = [2, 3, 4, 5, 5, 6, 7, 76]

print('choice    :', random.choice(liste))
print('random    :', random.random()) # flottant entre 0 et 1
print('randint   :', random.randint(5, 10)) # entier, bornes incluses
print('randrange :', random.randrange(100)) # entier, borne haute exclue
print('sample    :', random.sample(range(100), 5))

choice    : 5
random    : 0.7475160197943862
randint   : 7
randrange : 36
sample    : [98, 97, 80, 40, 82]


Attention à une incohérence de conception : `randint(5, 10)` inclut 10, alors
que `randrange(5, 10)` l'exclut comme `range`. Vérifie toujours laquelle tu
utilises.

`sample` tire sans remise pas de doublon. Pour tirer avec remise, c'est
`choices` avec un s :

```python
random.choices(liste, k=5) # peut répéter
random.sample(liste, k=5) # ne répète pas
```

### La reproductibilité

Sans graine, chaque exécution donne un résultat différent. Pour un notebook
publié, c'est un problème : le lecteur verra des chiffres qui ne correspondent
pas à tes commentaires.

In [25]:
random.seed(0)
print('après seed(0) :', [random.randint(1, 100) for _ in range(5)])

random.seed(0)
print('après seed(0) :', [random.randint(1, 100) for _ in range(5)])

print('sans reseed   :', [random.randint(1, 100) for _ in range(5)])

après seed(0) : [50, 98, 54, 6, 34]
après seed(0) : [50, 98, 54, 6, 34]
sans reseed   : [66, 63, 52, 39, 62]


Les deux premières lignes sont identiques, la troisième diffère. `seed` remet le
générateur dans un état connu.

Le point important : la graine agit sur toute la suite, pas sur une seule
cellule. Si tu la fixes en tête de notebook et que tu réexécutes une cellule du
milieu, tu n'obtiendras pas les mêmes valeurs qu'au premier passage.

Pour un notebook publié, mets `random.seed(0)` dans chaque cellule qui utilise
l'aléatoire, ou juste avant chaque tirage important.

Et attention : `random.seed` et `np.random.seed` sont indépendants. Fixer
l'un ne fixe pas l'autre.

### shuffle modifie sur place

In [26]:
random.seed(0)
liste = [2, 3, 4, 5, 5, 6, 7, 76]

print('avant  :', liste)
resultat = random.shuffle(liste)
print('après  :', liste)
print('retour :', resultat)

avant  : [2, 3, 4, 5, 5, 6, 7, 76]
après  : [5, 3, 6, 4, 2, 5, 76, 7]
retour : None


`shuffle` renvoie `None` et modifie la liste. C'est la convention vue au notebook
04 : `sort`, `append`, `shuffle` modifient sur place et ne renvoient rien.

L'erreur classique :

```python
liste = random.shuffle(liste) # liste devient None
```

Pour obtenir une copie mélangée sans toucher à l'original :

```python
melange = random.sample(liste, len(liste))
```

## 7. Les modules os et glob

`os` donne accès au système de fichiers.

In [27]:
import os
print('Dossier courant :', os.getcwd())
print()
print('Contenu :', os.listdir())

Dossier courant : /content

Contenu : ['.config', 'module_bavard.py', 'mes_outils.py', '__pycache__', 'sample_data']


`getcwd` signifie *get current working directory*. C'est le dossier depuis lequel
les chemins relatifs sont interprétés celui où `open('fichier.txt')` va
chercher.

Sur Colab, c'est `/content`.

In [28]:
# Construire un chemin correctement
chemin = os.path.join('donnees', 'brut', 'fichier.csv')
print(chemin)

print()
print('existe :', os.path.exists('mes_outils.py'))
print('est un fichier :', os.path.isfile('mes_outils.py'))
print('extension :', os.path.splitext('mes_outils.py'))

donnees/brut/fichier.csv

existe : True
est un fichier : True
extension : ('mes_outils', '.py')


`os.path.join` assemble un chemin avec le bon séparateur selon le système slash sur Linux et Mac, antislash sur Windows. Écrire `'donnees/brut/'` en dur
fonctionne souvent, mais casse sur Windows.

Pour créer un dossier :

```python
os.makedirs('resultats', exist_ok=True)
```

Le `exist_ok=True` évite une erreur si le dossier existe déjà.

### glob : chercher des fichiers par motif

In [29]:
import glob

print('Tout :', glob.glob('*'))
print('Les .py :', glob.glob('*.py'))
print('Les .txt :', glob.glob('*.txt'))

Tout : ['module_bavard.py', 'mes_outils.py', '__pycache__', 'sample_data']
Les .py : ['module_bavard.py', 'mes_outils.py']
Les .txt : []


L'étoile remplace n'importe quelle suite de caractères. Autres motifs :

| Motif | Correspond à |
|---|---|
| `*.csv` | tous les fichiers CSV |
| `data_*.csv` | ceux dont le nom commence par `data_` |
| `fichier?.txt` | un seul caractère variable |
| `**/*.csv` | récursif, avec `recursive=True` |

L'usage typique : traiter tous les fichiers d'un dossier.

In [30]:
# Créons quelques fichiers pour l'exemple
for i in range(3):
    with open(f'donnee_{i}.txt', 'w') as f:
        f.write(f'Contenu du fichier {i}\n')

fichiers = glob.glob('donnee_*.txt')
print('Trouvés :', fichiers)
print()

for chemin in sorted(fichiers): # sorted : ordre garanti
    with open(chemin, 'r') as f:
        print(f'{chemin} -> {f.read().strip()}')

Trouvés : ['donnee_0.txt', 'donnee_2.txt', 'donnee_1.txt']

donnee_0.txt -> Contenu du fichier 0
donnee_1.txt -> Contenu du fichier 1
donnee_2.txt -> Contenu du fichier 2


Le `sorted` n'est pas décoratif. `glob` ne garantit aucun ordre il dépend du
système de fichiers. Sans tri, l'ordre de traitement peut changer d'une machine
à l'autre, ce qui rend un traitement non reproductible.

C'est le même genre de problème que l'ordre d'un ensemble au notebook 06.

Un dernier détail : `glob('*')` ignore les fichiers cachés commençant par un
point. Pour les inclure, il faut `glob('.*')` en plus.

### Faire le ménage

In [31]:
for chemin in glob.glob('donnee_*.txt'):
    os.remove(chemin)

print('Restant :', glob.glob('donnee_*.txt'))

Restant : []


## 8. Un mot sur les variables écrasées

Ce notebook utilise `liste` pour plusieurs choses différentes : une liste de
notes pour `statistics`, la même mélangée par `shuffle`, puis parfois le
résultat d'un `glob`.

C'est le piège récurrent depuis le début. `liste = glob.glob('*')` remplace
silencieusement les données statistiques par des noms de fichiers, et toute
cellule qui remonte plus haut travaille alors sur autre chose.

La parade tient en deux règles :

Un nom par usage. `notes` pour les statistiques, `fichiers` pour le glob.

Et avant toute publication : redémarrer le noyau, tout réexécuter dans l'ordre.
C'est le seul moyen de voir ces problèmes.

In [32]:
notes = [2, 3, 4, 5, 5, 6, 7, 76]
fichiers = glob.glob('*.py')

print('notes :', notes)
print('fichiers :', fichiers)

notes : [2, 3, 4, 5, 5, 6, 7, 76]
fichiers : ['module_bavard.py', 'mes_outils.py']


## 9. Mémo

### Les modules du notebook

| Module | Contenu |
|---|---|
| `math` | constantes, trigonométrie, exponentielle, arrondis directionnels |
| `statistics` | moyenne, médiane, mode, variance |
| `random` | tirages aléatoires |
| `os` | système de fichiers, chemins |
| `glob` | recherche de fichiers par motif |

### Les formes d'import

| Forme | Usage |
|---|---|
| `import module` | par défaut |
| `import module as m` | noms longs, conventions `np`, `pd`, `plt` |
| `from module import x` | une ou deux fonctions |
| `from module import *` | à éviter |

### Les pièges

| Situation | Ce qui se passe |
|---|---|
| Modifier un module déjà importé | l'ancienne version reste, il faut `reload` |
| `from module import *` | écrase les variables du même nom |
| Nommer un fichier `random.py` | masque le module standard |
| `statistics.variance` vs `np.var` | n-1 contre n, résultats différents |
| `random` sans `seed` | notebook non reproductible |
| `random.shuffle(l)` réassigné | la liste devient `None` |
| `glob` sans `sorted` | ordre non garanti |
| `randint(5, 10)` | inclut 10, contrairement à `range` |

## 10. Exercices

**Exercice 1**

Écris un module `outils_texte.py` contenant trois fonctions : compter les mots
d'une phrase, inverser une chaîne, et vérifier si un mot est un palindrome.
Ajoute une docstring à chacune et une section `if __name__ == '__main__'` qui
teste les trois.

Importe ensuite le module et vérifie que les tests ne s'exécutent pas.

**Exercice 2**

Écris une fonction `simuler_des(n, graine=None)` qui lance deux dés n fois et
renvoie un dictionnaire des fréquences de chaque somme, de 2 à 12. Vérifie que
deux appels avec la même graine donnent le même résultat.

Compare ensuite la distribution obtenue sur 100 lancers et sur 100 000. Laquelle
approche le mieux la théorie, et pourquoi la somme 7 est-elle la plus fréquente ?

**Exercice 3**

Écris un script qui crée dix fichiers `mesure_00.txt` à `mesure_09.txt`, chacun
contenant cinq nombres aléatoires. Puis relis-les tous avec `glob`, calcule la
moyenne de chaque fichier et la moyenne globale.

Attention à deux choses : l'ordre de `glob`, et le fait que ton script doit
pouvoir être relancé sans accumuler les fichiers.

## Pour continuer

Le notebook suivant porte sur la programmation orientée objet : les classes, les
attributs et les méthodes. C'est ce qui explique pourquoi on écrit
`liste.append()` avec un point.

Puis viendra NumPy, où `math` et `statistics` seront remplacés par des versions
vectorisées, capables de travailler sur des millions de valeurs à la fois.

In [33]:
# Exercice 1


In [34]:
%%writefile outils_texte.py

import string

def compter_mots(phrase):
    """Retourne le nombre de mots d'une phrase.
    Les mots sont séparés par des espaces, tabulations ou sauts de ligne.
    """
    return len(phrase.split())

def inverser(chaine):
    """Retourne la chaîne à l'envers."""
    return chaine[::-1]

def est_palindrome(mot):
    """Indique si un mot ou une phrase se lit pareil dans les deux sens.
    La casse, les espaces et la ponctuation sont ignorés.
    """
    propre = mot.lower()
    for signe in string.punctuation + ' ':
        propre = propre.replace(signe, '')
    return propre == propre[::-1]

if __name__ == '__main__':
    print('Tests du module')
    print(compter_mots('un deux trois')) # 3
    print(inverser('abc')) # cba
    print(est_palindrome('Kayak')) # True
    print(est_palindrome('Engage le jeu que je le gagne')) # True
    print(est_palindrome('bonjour')) # False

Writing outils_texte.py


In [35]:
import outils_texte

print('Rien ne s\'est affiché au-dessus')
print(outils_texte.est_palindrome('Kayak'))
print('__name__ vaut :', outils_texte.__name__)

Rien ne s'est affiché au-dessus
True
__name__ vaut : outils_texte


In [36]:
!python outils_texte.py

Tests du module
3
cba
True
True
False


In [37]:
# Exercice 2

In [38]:
import random

def simuler_des(n, graine=None):
    """Lance deux dés n fois et compte la fréquence de chaque somme.
    graine : si fournie, rend le tirage reproductible.
    Retourne un dictionnaire {somme: nombre d'occurrences}, de 2 à 12.
    """
    if graine is not None:
        random.seed(graine)

    frequences = {somme: 0 for somme in range(2, 13)}

    for _ in range(n):
        total = random.randint(1, 6) + random.randint(1, 6)
        frequences[total] += 1

    return frequences

In [39]:
a = simuler_des(1000, graine=0)
b = simuler_des(1000, graine=0)
c = simuler_des(1000) # sans graine

print('a == b :', a == b)
print('a == c :', a == c)
a

a == b : True
a == c : False


{2: 27,
 3: 53,
 4: 94,
 5: 114,
 6: 138,
 7: 164,
 8: 149,
 9: 103,
 10: 76,
 11: 52,
 12: 30}

In [40]:
theorie = {s: (6 - abs(7 - s)) / 36 for s in range(2, 13)}

for n in [100, 100_000,1000000]:
    obs = simuler_des(n, graine=0)
    print(f'\n- {n} lancers-')
    for s in range(2, 13):
        reel = obs[s] / n
        print(f'{s:>3} : observé {reel:.3%}   théorie {theorie[s]:.3%}   '
              f'écart {abs(reel - theorie[s]):.3%}')


- 100 lancers-
  2 : observé 3.000%   théorie 2.778%   écart 0.222%
  3 : observé 5.000%   théorie 5.556%   écart 0.556%
  4 : observé 9.000%   théorie 8.333%   écart 0.667%
  5 : observé 12.000%   théorie 11.111%   écart 0.889%
  6 : observé 13.000%   théorie 13.889%   écart 0.889%
  7 : observé 22.000%   théorie 16.667%   écart 5.333%
  8 : observé 13.000%   théorie 13.889%   écart 0.889%
  9 : observé 10.000%   théorie 11.111%   écart 1.111%
 10 : observé 4.000%   théorie 8.333%   écart 4.333%
 11 : observé 7.000%   théorie 5.556%   écart 1.444%
 12 : observé 2.000%   théorie 2.778%   écart 0.778%

- 100000 lancers-
  2 : observé 2.834%   théorie 2.778%   écart 0.056%
  3 : observé 5.525%   théorie 5.556%   écart 0.031%
  4 : observé 8.466%   théorie 8.333%   écart 0.133%
  5 : observé 11.192%   théorie 11.111%   écart 0.081%
  6 : observé 13.809%   théorie 13.889%   écart 0.080%
  7 : observé 17.036%   théorie 16.667%   écart 0.369%
  8 : observé 13.672%   théorie 13.889%   écart 

In [41]:
combinaisons = {}
for d1 in range(1, 7):
    for d2 in range(1, 7):
        somme = d1 + d2
        combinaisons[somme] = combinaisons.get(somme, 0) + 1

for s in sorted(combinaisons):
    print(f'{s:>3} : {combinaisons[s]} combinaisons sur 36  '
          f'-> {combinaisons[s]/36:.2%}')

  2 : 1 combinaisons sur 36  -> 2.78%
  3 : 2 combinaisons sur 36  -> 5.56%
  4 : 3 combinaisons sur 36  -> 8.33%
  5 : 4 combinaisons sur 36  -> 11.11%
  6 : 5 combinaisons sur 36  -> 13.89%
  7 : 6 combinaisons sur 36  -> 16.67%
  8 : 5 combinaisons sur 36  -> 13.89%
  9 : 4 combinaisons sur 36  -> 11.11%
 10 : 3 combinaisons sur 36  -> 8.33%
 11 : 2 combinaisons sur 36  -> 5.56%
 12 : 1 combinaisons sur 36  -> 2.78%


In [42]:
# Exerice 3

In [43]:
import os
import glob
import random

def creer_mesures(nb_fichiers=10, nb_valeurs=5, graine=0):
    """Crée des fichiers de mesures, en supprimant les précédents."""
    # Nettoyage : le script doit pouvoir être relancé
    for ancien in glob.glob('mesure_*.txt'):
        os.remove(ancien)

    random.seed(graine)

    for i in range(nb_fichiers):
        nom = f'mesure_{i:02d}.txt' # 02d : deux chiffres, zéro devant
        with open(nom, 'w') as f:
            for _ in range(nb_valeurs):
                f.write(f'{random.uniform(0, 100):.2f}\n')

    return sorted(glob.glob('mesure_*.txt'))


fichiers = creer_mesures()
print(fichiers)

['mesure_00.txt', 'mesure_01.txt', 'mesure_02.txt', 'mesure_03.txt', 'mesure_04.txt', 'mesure_05.txt', 'mesure_06.txt', 'mesure_07.txt', 'mesure_08.txt', 'mesure_09.txt']


In [44]:
def lire_mesures():
    """Lit tous les fichiers de mesures et calcule les moyennes."""
    fichiers = sorted(glob.glob('mesure_*.txt')) # sorted obligatoire

    if not fichiers:
        print('Aucun fichier trouvé')
        return None

    toutes = []
    print(f'{"fichier":<16} {"n":>3} {"moyenne":>9}')

    for chemin in fichiers:
        with open(chemin) as f:
            valeurs = [float(l) for l in f if l.strip()]

        toutes.extend(valeurs) # extend, pas append
        print(f'{chemin:<16} {len(valeurs):>3} {sum(valeurs)/len(valeurs):>9.2f}')

    print()
    print(f'Total : {len(toutes)} valeurs, moyenne globale {sum(toutes)/len(toutes):.2f}')
    return toutes

toutes = lire_mesures()

fichier            n   moyenne
mesure_00.txt      5     55.86
mesure_01.txt      5     51.04
mesure_02.txt      5     61.38
mesure_03.txt      5     77.11
mesure_04.txt      5     61.90
mesure_05.txt      5     60.51
mesure_06.txt      5     59.13
mesure_07.txt      5     52.51
mesure_08.txt      5     38.63
mesure_09.txt      5     56.70

Total : 50 valeurs, moyenne globale 57.48


In [45]:
creer_mesures(nb_fichiers=3)
print('Après relance avec 3 fichiers :', sorted(glob.glob('mesure_*.txt')))

for chemin in glob.glob('mesure_*.txt'):
    os.remove(chemin)

Après relance avec 3 fichiers : ['mesure_00.txt', 'mesure_01.txt', 'mesure_02.txt']
